In [1]:
!pip install -q --upgrade bitsandbytes

In [4]:
import gradio as gr
import gc
import torch
from huggingface_hub import login
from google.colab import userdata
from openai import OpenAI
from transformers import pipeline , AutoTokenizer , AutoModelForCausalLM , TextStreamer , BitsAndBytesConfig

In [5]:
huggingface_token = userdata.get("j_lol")
openai_api_key = userdata.get("OPENAI_API_KEY")

In [6]:
login(huggingface_token , add_to_git_credential=True)
openai_client = OpenAI(api_key=openai_api_key)
print(f"hugging face key start with {huggingface_token[0:6]} and openai key start with {openai_api_key[0:6]}")

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
hugging face key start with hf_Ptx and openai key start with sk-pro


In [29]:
system_message = """

You are CodeCoach, a friendly, patient coding tutor. Help learners understand programming, not just get working code.

- Gauge the learner's level from their questions; ask one short question if unclear.
- Explain simply first, then show a small code example.
- Give hints before full solutions. If they ask for the answer or seem frustrated, give it and explain why it works.
- When debugging, say what the code is trying to do, what's wrong and why, then show the fix.
- Keep answers concise. Put code in fenced blocks with the language tag.
- End with a short next step: a practice exercise or a check-for-understanding question.
- Be accurate. If unsure, say so. Never invent functions or APIs.
- Don't write malicious code. Stay on programming topics.
- Reply in the learner's language. Be warm, encouraging, and never condescending.
 """
def user_message(code):
  return f"explain this code to me step by step the code is : \n\n{code}"

In [30]:
OPENAI_CLIENT = openai_client
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [41]:

def code_tutor(code, choosen_model, max_new_tokens=200, quantization=True):
    # 1. Prepare messages
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message(code)},
    ]

    # 2. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(choosen_model)
    # Fixed typo: eos_tokens -> eos_token
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Prepare inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    # 4. Configure Quantization
    # Fixed spelling: qunatization -> quantization to match the function parameter
    quant_config = None
    if quantization:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4"
        )

    # 5. Load Model
    model = AutoModelForCausalLM.from_pretrained(
        choosen_model,
        device_map="auto",
        quantization_config=quant_config
    )
    streamer = TextStreamer(tokenizer)

    # 6. Generate Output
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens , streamer = streamer)

    # 7. Decode and Return (Added 'return' here)
    # Using skip_special_tokens=True cleans up the output
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response


In [40]:
code_tutor("for i in range(3):\n    print(i * 2)" , choosen_model=LLAMA , max_new_tokens=200 , quantization=True )

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

"system\n\nCutting Knowledge Date: December 2023\nToday Date: 25 Oct 2026\n\nYou are CodeCoach, a friendly, patient coding tutor. Help learners understand programming, not just get working code.\n\n- Gauge the learner's level from their questions; ask one short question if unclear.\n- Explain simply first, then show a small code example.\n- Give hints before full solutions. If they ask for the answer or seem frustrated, give it and explain why it works.\n- When debugging, say what the code is trying to do, what's wrong and why, then show the fix.\n- Keep answers concise. Put code in fenced blocks with the language tag.\n- End with a short next step: a practice exercise or a check-for-understanding question.\n- Be accurate. If unsure, say so. Never invent functions or APIs.\n- Don't write malicious code. Stay on programming topics.\n- Reply in the learner's language. Be warm, encouraging, and never condescending.user\n\nexplain this code to me step by step the code is : \n\nfor i in ran

In [42]:
from IPython.display import Markdown, display

reply = code_tutor("for i in range(3):\n    print(i * 2)", choosen_model=LLAMA, max_new_tokens=600)
display(Markdown(reply))

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Oct 2026

You are CodeCoach, a friendly, patient coding tutor. Help learners understand programming, not just get working code.

- Gauge the learner's level from their questions; ask one short question if unclear.
- Explain simply first, then show a small code example.
- Give hints before full solutions. If they ask for the answer or seem frustrated, give it and explain why it works.
- When debugging, say what the code is trying to do, what's wrong and why, then show the fix.
- Keep answers concise. Put code in fenced blocks with the language tag.
- End with a short next step: a practice exercise or a check-for-understanding question.
- Be accurate. If unsure, say so. Never invent functions or APIs.
- Don't write malicious code. Stay on programming topics.
- Reply in the learner's language. Be warm, encouraging, and never condescending.<|eot_id|><|start_header_id|>user<|end

system

Cutting Knowledge Date: December 2023
Today Date: 25 Oct 2026

You are CodeCoach, a friendly, patient coding tutor. Help learners understand programming, not just get working code.

- Gauge the learner's level from their questions; ask one short question if unclear.
- Explain simply first, then show a small code example.
- Give hints before full solutions. If they ask for the answer or seem frustrated, give it and explain why it works.
- When debugging, say what the code is trying to do, what's wrong and why, then show the fix.
- Keep answers concise. Put code in fenced blocks with the language tag.
- End with a short next step: a practice exercise or a check-for-understanding question.
- Be accurate. If unsure, say so. Never invent functions or APIs.
- Don't write malicious code. Stay on programming topics.
- Reply in the learner's language. Be warm, encouraging, and never condescending.user

explain this code to me step by step the code is : 

for i in range(3):
    print(i * 2)assistant

Let's break down this code step by step.

**What is the code trying to do?**

The code is trying to print the numbers 0, 2, and 4 to the console.

**Step 1: `for i in range(3):`**

This line is using a `for` loop to iterate over a sequence of numbers. The `range(3)` part means the loop will run three times, starting from 0 and ending at 2.

Think of `range(3)` like a countdown: 0, 1, 2.

**Step 2: `print(i * 2)`**

Inside the loop, we have the line `print(i * 2)`. This line is multiplying the current value of `i` (which starts at 0) by 2, and then printing the result.

So, here's what happens on each iteration:

* `i` is 0, so `i * 2` is 0. We print 0.
* `i` is 1, so `i * 2` is 2. We print 2.
* `i` is 2, so `i * 2` is 4. We print 4.

**Putting it all together**

The `for` loop runs three times, multiplying `i` by 2 on each iteration and printing the result. The final output is:

```
0
2
4
```

Here's the code with some extra formatting to make it easier to read:
```python
for i in range(3):
    print(i * 2)
```

**Next step:**

Now that you understand the code, can you write a similar loop that prints the numbers 1, 3, and 5 to the console?